## **Aim**
To implement a program that detects unauthorized USB device connections using a simulated Windows USB activity log.

## **Algorithm**
**Step 1:** Import `json`, `datetime`, `collections`, and `re` libraries.

**Step 2:** Create a simulated Windows USB activity log (JSON) based on Windows Event Logs (Event IDs 2003, 2004, 2005, 2006, 2010, 2100, 2101, 2102) and SetupAPI logs.

**Step 3:** Define fields: timestamp, event_id, device_id, vendor_id, product_id, device_class, device_description, serial_number, drive_letter, user.

**Step 4:** Define authorized device list (whitelist) by serial number or VID/PID.

**Step 5:** Parse log and identify:
   - New device connections not in whitelist
   - Mass storage devices (potential data exfiltration)
   - Devices connected at unusual hours
   - Multiple devices connected rapidly
   - Devices with suspicious VID/PID (e.g., rubber ducky, bash bunny)

**Step 6:** Generate a report of unauthorized USB events.

In [1]:
import json
from datetime import datetime, timedelta
from collections import defaultdict, Counter
import os

AUTHORIZED_DEVICES = {
    "USBSTOR\\DISK&VEN_KINGSTON&PROD_DATATRAVELER_3.0&REV_1.00\\1234567890ABCDEF&0": "Company Kingston USB",
    "USBSTOR\\DISK&VEN_SANDISK&PROD_ULTRA_FIT&REV_1.00\\9876543210FEDCBA&0": "Company SanDisk USB",
    "VID_046D&PID_C52B": "Logitech Unifying Receiver",
    "VID_045E&PID_00DB": "Microsoft Keyboard",
}

SUSPICIOUS_VID_PID = {
    ("1B4F", "0001"): "Hak5 USB Rubber Ducky",
    ("1B4F", "0002"): "Hak5 Bash Bunny",
    ("0483", "5740"): "STM32 HID Bootloader (BadUSB)",
    ("16C0", "05DC"): "Teensyduino (BadUSB capable)",
    ("239A", "001A"): "Adafruit CircuitPython (HID capable)",
}

DEVICE_CLASSES = {
    "01": "Audio",
    "02": "Communications",
    "03": "HID (Human Interface Device)",
    "05": "Physical",
    "06": "Image",
    "07": "Printer",
    "08": "Mass Storage",  # High risk for data exfil
    "09": "Hub",
    "0A": "CDC-Data",
    "0B": "Smart Card",
    "0D": "Content Security",
    "0E": "Video",
    "0F": "Personal Healthcare",
    "10": "Audio/Video",
    "11": "Billboard",
    "12": "USB Type-C Bridge",
    "DC": "Diagnostic",
    "E0": "Wireless Controller",
    "EF": "Miscellaneous",
    "FE": "Application Specific",
    "FF": "Vendor Specific",
}

def create_sample_usb_log(log_file):
    now = datetime.now()
    base = now - timedelta(hours=48)
    
    events = [
        # Authorized device - normal work hours
        {"timestamp": (base + timedelta(hours=2)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "USBSTOR\\DISK&VEN_KINGSTON&PROD_DATATRAVELER_3.0&REV_1.00\\1234567890ABCDEF&0",
         "vid": "0951", "pid": "1666", "class": "08", "description": "Kingston DataTraveler 3.0",
         "serial": "1234567890ABCDEF", "drive_letter": "E:", "user": "user"},
        {"timestamp": (base + timedelta(hours=2, minutes=30)).isoformat(), "event_id": 2006, "action": "DEVICE_REMOVAL",
         "device_id": "USBSTOR\\DISK&VEN_KINGSTON&PROD_DATATRAVELER_3.0&REV_1.00\\1234567890ABCDEF&0",
         "drive_letter": "E:", "user": "user"},
        
        # Authorized device - mouse/keyboard
        {"timestamp": (base + timedelta(hours=8)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "HID\\VID_046D&PID_C52B\\6&12345678&0&0000",
         "vid": "046D", "pid": "C52B", "class": "03", "description": "Logitech Wireless Mouse",
         "serial": "", "drive_letter": "", "user": "user"},
        
        # Unauthorized mass storage - potential exfiltration
        {"timestamp": (base + timedelta(hours=24)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "USBSTOR\\DISK&VEN_GENERIC&PROD_USB_FLASH_DRIVE&REV_1.00\\ABCDEF1234567890&0",
         "vid": "1234", "pid": "5678", "class": "08", "description": "Generic USB Flash Drive",
         "serial": "ABCDEF1234567890", "drive_letter": "F:", "user": "user"},
        {"timestamp": (base + timedelta(hours=24, minutes=5)).isoformat(), "event_id": 2010, "action": "VOLUME_MOUNT",
         "device_id": "USBSTOR\\DISK&VEN_GENERIC&PROD_USB_FLASH_DRIVE&REV_1.00\\ABCDEF1234567890&0",
         "drive_letter": "F:", "user": "user"},
        {"timestamp": (base + timedelta(hours=24, minutes=45)).isoformat(), "event_id": 2006, "action": "DEVICE_REMOVAL",
         "device_id": "USBSTOR\\DISK&VEN_GENERIC&PROD_USB_FLASH_DRIVE&REV_1.00\\ABCDEF1234567890&0",
         "drive_letter": "F:", "user": "user"},
        
        # Suspicious HID device - Rubber Ducky
        {"timestamp": (base + timedelta(hours=30)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "HID\\VID_1B4F&PID_0001\\6&87654321&0&0000",
         "vid": "1B4F", "pid": "0001", "class": "03", "description": "USB Keyboard",
         "serial": "", "drive_letter": "", "user": "user"},
        
        # Another unauthorized mass storage at night
        {"timestamp": (now - timedelta(hours=2)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "USBSTOR\\DISK&VEN_SAMSUNG&PROD_PORTABLE_SSD_T7&REV_1.00\\SAMSUNG123456789&0",
         "vid": "04E8", "pid": "61F5", "class": "08", "description": "Samsung Portable SSD T7",
         "serial": "SAMSUNG123456789", "drive_letter": "G:", "user": "user"},
        {"timestamp": (now - timedelta(hours=1, minutes=30)).isoformat(), "event_id": 2006, "action": "DEVICE_REMOVAL",
         "device_id": "USBSTOR\\DISK&VEN_SAMSUNG&PROD_PORTABLE_SSD_T7&REV_1.00\\SAMSUNG123456789&0",
         "drive_letter": "G:", "user": "user"},
        
        # Multiple rapid connections (USB drop attack simulation)
        {"timestamp": (now - timedelta(minutes=30)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "USBSTOR\\DISK&VEN_UNKNOWN&PROD_MALICIOUS&REV_1.00\\MAL001&0",
         "vid": "DEAD", "pid": "BEEF", "class": "08", "description": "Unknown Device",
         "serial": "MAL001", "drive_letter": "H:", "user": "user"},
        {"timestamp": (now - timedelta(minutes=29)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "HID\\VID_16C0&PID_05DC\\6&11111111&0&0000",
         "vid": "16C0", "pid": "05DC", "class": "03", "description": "Teensyduino USB Device",
         "serial": "", "drive_letter": "", "user": "user"},
        {"timestamp": (now - timedelta(minutes=28)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "USBSTOR\\DISK&VEN_UNKNOWN&PROD_MALICIOUS2&REV_1.00\\MAL002&0",
         "vid": "DEAD", "pid": "BEEF", "class": "08", "description": "Unknown Device 2",
         "serial": "MAL002", "drive_letter": "I:", "user": "user"},
        
        # Legitimate printer
        {"timestamp": (base + timedelta(hours=10)).isoformat(), "event_id": 2003, "action": "DEVICE_ARRIVAL",
         "device_id": "USBPRINT\\HP_LASERJET_1234\\6&22222222&0&0000",
         "vid": "03F0", "pid": "1234", "class": "07", "description": "HP LaserJet Pro",
         "serial": "PRINTER123", "drive_letter": "", "user": "user"},
    ]
    
    with open(log_file, "w") as f:
        json.dump(events, f, indent=2)

def analyze_usb_log(log_file):
    with open(log_file, "r") as f:
        events = json.load(f)
    
    arrivals = [e for e in events if e["action"] == "DEVICE_ARRIVAL"]
    
    alerts = []
    
    # Check each arrival
    for e in arrivals:
        device_id = e["device_id"]
        vid = e["vid"].upper()
        pid = e["pid"].upper()
        class_code = e["class"]
        serial = e.get("serial", "")
        timestamp = datetime.fromisoformat(e["timestamp"])
        hour = timestamp.hour
        
        # Check if authorized
        authorized = False
        auth_name = ""
        for auth_id, name in AUTHORIZED_DEVICES.items():
            if auth_id in device_id or (auth_id.startswith("VID_") and f"VID_{vid}&PID_{pid}" in auth_id):
                authorized = True
                auth_name = name
                break
        
        # Check suspicious VID/PID
        suspicious_device = None
        for (svid, spid), name in SUSPICIOUS_VID_PID.items():
            if vid == svid.upper() and pid == spid.upper():
                suspicious_device = name
                break
        
        # Mass storage class
        is_mass_storage = class_code == "08"
        class_name = DEVICE_CLASSES.get(class_code, f"Unknown ({class_code})")
        
        # Off-hours (before 6am or after 10pm)
        off_hours = hour < 6 or hour > 22
        
        alert_info = {
            "timestamp": e["timestamp"],
            "device_id": device_id,
            "description": e["description"],
            "vid_pid": f"{vid}:{pid}",
            "class": f"{class_code} ({class_name})",
            "serial": serial or "N/A",
            "drive_letter": e.get("drive_letter", "N/A"),
            "authorized": authorized,
            "auth_name": auth_name,
            "suspicious_vid_pid": suspicious_device,
            "is_mass_storage": is_mass_storage,
            "off_hours": off_hours,
            "user": e["user"],
        }
        
        alerts.append(alert_info)
    
    # Detect rapid multiple connections (within 5 minutes)
    arrivals_sorted = sorted(arrivals, key=lambda x: datetime.fromisoformat(x["timestamp"]))
    for i in range(len(arrivals_sorted) - 1):
        t1 = datetime.fromisoformat(arrivals_sorted[i]["timestamp"])
        t2 = datetime.fromisoformat(arrivals_sorted[i+1]["timestamp"])
        if (t2 - t1).total_seconds() < 300:  # 5 minutes
            alerts.append({
                "type": "RAPID_CONNECTIONS",
                "timestamp": arrivals_sorted[i]["timestamp"],
                "device1": arrivals_sorted[i]["description"],
                "device2": arrivals_sorted[i+1]["description"],
                "interval_seconds": int((t2 - t1).total_seconds()),
            })
    
    return alerts, arrivals

def main():
    log_file = "usb_activity_log.json"
    create_sample_usb_log(log_file)
    
    print("Analyzing USB activity log...")
    alerts, arrivals = analyze_usb_log(log_file)
    
    print(f"\n{'='*70}")
    print(f"USB DEVICE CONNECTION ANALYSIS")
    print(f"{'='*70}")
    print(f"Total device arrivals: {len(arrivals)}")
    
    print(f"\n--- ALL USB CONNECTIONS ---")
    print(f"{'Time':<20} {'Device':<35} {'VID:PID':<10} {'Class':<20} {'Auth':<8} {'Drive'}")
    print("-" * 100)
    for a in alerts:
        if "type" in a:
            continue
        auth_status = "YES" if a["authorized"] else "NO"
        print(f"{a['timestamp'][:19]:<20} {a['description'][:34]:<35} {a['vid_pid']:<10} {a['class'][:19]:<20} {auth_status:<8} {a['drive_letter']}")
    
    print(f"\n--- UNAUTHORIZED / SUSPICIOUS DEVICES ---")
    suspicious_found = False
    for a in alerts:
        if "type" in a:
            continue
        if not a["authorized"] or a["suspicious_vid_pid"] or a["off_hours"]:
            suspicious_found = True
            print(f"\n  ⚠ DEVICE: {a['description']} ({a['vid_pid']})")
            print(f"     Time: {a['timestamp']}")
            print(f"     Device ID: {a['device_id']}")
            print(f"     Class: {a['class']}")
            print(f"     Serial: {a['serial']}")
            print(f"     Drive Letter: {a['drive_letter']}")
            print(f"     User: {a['user']}")
            print(f"     Authorized: {'YES - ' + a['auth_name'] if a['authorized'] else 'NO'}")
            if a["suspicious_vid_pid"]:
                print(f"     ⚠ KNOWN MALICIOUS DEVICE: {a['suspicious_vid_pid']}")
            if a["is_mass_storage"]:
                print(f"     ⚠ MASS STORAGE DEVICE - Data exfiltration risk")
            if a["off_hours"]:
                print(f"     ⚠ OFF-HOURS CONNECTION")
    
    if not suspicious_found:
        print("  No unauthorized devices detected.")
    
    print(f"\n--- RAPID MULTIPLE CONNECTIONS ---")
    rapid = [a for a in alerts if a.get("type") == "RAPID_CONNECTIONS"]
    if rapid:
        for r in rapid:
            print(f"  ⚠ {r['timestamp']}: {r['device1']} -> {r['device2']} ({r['interval_seconds']}s apart)")
    else:
        print("  No rapid connection sequences detected.")
    
    # Summary stats
    print(f"\n--- SUMMARY ---")
    total = len(arrivals)
    unauthorized = sum(1 for a in alerts if "type" not in a and not a["authorized"])
    mass_storage = sum(1 for a in alerts if "type" not in a and a["is_mass_storage"])
    off_hours = sum(1 for a in alerts if "type" not in a and a["off_hours"])
    known_bad = sum(1 for a in alerts if "type" not in a and a["suspicious_vid_pid"])
    
    print(f"  Total connections: {total}")
    print(f"  Unauthorized devices: {unauthorized}")
    print(f"  Mass storage devices: {mass_storage}")
    print(f"  Off-hours connections: {off_hours}")
    print(f"  Known malicious devices: {known_bad}")

if __name__ == "__main__":
    main()

Analyzing USB activity log...

USB DEVICE CONNECTION ANALYSIS
Total device arrivals: 9

--- ALL USB CONNECTIONS ---
Time                 Device                              VID:PID    Class                Auth     Drive
----------------------------------------------------------------------------------------------------
2026-08-18T11:14:40  Kingston DataTraveler 3.0           0951:1666  08 (Mass Storage)    YES      E:
2026-08-18T17:14:40  Logitech Wireless Mouse             046D:C52B  03 (HID (Human Inte  YES      
2026-08-19T09:14:40  Generic USB Flash Drive             1234:5678  08 (Mass Storage)    NO       F:
2026-08-19T15:14:40  USB Keyboard                        1B4F:0001  03 (HID (Human Inte  NO       
2026-08-20T07:14:40  Samsung Portable SSD T7             04E8:61F5  08 (Mass Storage)    NO       G:
2026-08-20T08:44:40  Unknown Device                      DEAD:BEEF  08 (Mass Storage)    NO       H:
2026-08-20T08:45:40  Teensyduino USB Device              16C0:05DC  03 (HID (

## **Result**
This the program successfully detects unauthorized USB device connections using a simulated Windows USB activity log.